# Entraînement intermédiaire — mini-MIAS

> ### ⚠️ Modèle de transition, pas un modèle clinique
>
> Ce notebook entraîne un classifieur bénin / malin sur **mini-MIAS**, en
> attendant l'accès à Mini-DDSM. mini-MIAS contient 322 clichés, dont
> **115 seulement portent une lésion annotée** — c'est ce sous-ensemble qui
> entraîne le modèle (voir l'étape 2). C'est un ordre de grandeur en dessous de
> ce qu'exige un classifieur de dépistage, et plus limité encore que le futur
> modèle Mini-DDSM.
>
> Le modèle produit ici sert à remplacer le placeholder par quelque chose qui a
> au moins vu des mammographies. Il **ne doit fonder aucune décision clinique**
> et doit être remplacé dès que Mini-DDSM est disponible.

## Ce que le notebook garantit

| Point | Garantie |
|-------|----------|
| Prétraitement | Importé de `app.ai.preprocessing`, pas réimplémenté ici |
| Ordre des classes | Importé de `app.ai.CLASS_NAMES`, jamais écrit en dur |
| Découpage | Par **patiente**, jamais par image (les clichés MIAS vont par paires) |
| Checkpoint | Relu par `app.ai.inference.loader.load_checkpoint` avant la fin du notebook |

## Données attendues

```
<MIAS_ROOT>/
├── all-mias/       # mdb001.pgm … mdb322.pgm
└── Info.txt        # REFNUM BG CLASS SEVERITY X Y RADIUS
```

`Info.txt` existe aussi dans `all-mias/` : le notebook accepte les deux
emplacements.

## Configuration

Les valeurs se surchargent par variables d'environnement, ce qui permet de
rejouer le notebook en local sur un échantillon (`BREASTAI_SMOKE=1`) avant de le
lancer sur Colab.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def _default_repo_root() -> Path:
    """Remonte jusqu'au dossier contenant `backend/app/ai`."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        if (candidate / "backend" / "app" / "ai").is_dir():
            return candidate
    return Path("/content/BreastAi") if IN_COLAB else here


# Dépôt : sur Colab il est cloné à l'étape suivante, en local il est déjà là.
REPO_ROOT = Path(os.environ.get("BREASTAI_REPO_ROOT", _default_repo_root()))
REPO_URL = os.environ.get("BREASTAI_REPO_URL", "https://github.com/MAHAMAT767/BreastAi.git")

# Dataset mini-MIAS.
MIAS_ROOT = Path(
    os.environ.get(
        "BREASTAI_MIAS_ROOT",
        "/content/drive/MyDrive/mias" if IN_COLAB else r"C:\Dev\datasets\mias",
    )
)

# Sorties : checkpoint et fiche modèle dans `models/`, images converties dans
# `datasets/`. Les deux dossiers sont exclus du dépôt par .gitignore — des
# mammographies n'ont rien à faire dans un commit.
OUTPUT_DIR = Path(os.environ.get("BREASTAI_OUTPUT_DIR", REPO_ROOT / "models"))
PNG_CACHE_DIR = Path(
    os.environ.get("BREASTAI_PNG_CACHE", REPO_ROOT / "datasets" / "mias_png")
)

# Mode vérification : quelques images, une époque. Sert à valider la mécanique
# du notebook, pas à produire un modèle.
SMOKE_TEST = os.environ.get("BREASTAI_SMOKE", "0") == "1"

SEED = 20260814
ARCHITECTURE = "efficientnet_b0"
MODEL_VERSION = os.environ.get("BREASTAI_MODEL_VERSION", "efficientnet_b0-mini-mias-v1")

BATCH_SIZE = 4 if SMOKE_TEST else 16
EPOCHS = 1 if SMOKE_TEST else 30
EARLY_STOPPING_PATIENCE = 1 if SMOKE_TEST else 8
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4
WEIGHT_DECAY = 1e-4

# Sensibilité visée pour le choix du seuil : en dépistage, un faux négatif coûte
# bien plus cher qu'un faux positif.
TARGET_SENSITIVITY = 0.90

# Nombre d'images retenues en mode vérification.
SMOKE_MAX_IMAGES = int(os.environ.get("BREASTAI_SMOKE_IMAGES", "24"))

for name, value in [
    ("REPO_ROOT", REPO_ROOT),
    ("MIAS_ROOT", MIAS_ROOT),
    ("OUTPUT_DIR", OUTPUT_DIR),
    ("SMOKE_TEST", SMOKE_TEST),
]:
    print(f"{name:12} = {value}")

### Environnement Colab

Sur Colab : montage de Drive, clonage du dépôt, installation des dépendances
d'imagerie. En local, cette cellule ne fait rien.

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    if not (REPO_ROOT / "backend" / "app" / "ai").is_dir():
        # `--depth 1` : seul l'état courant du code nous intéresse.
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")

    # torch, torchvision et numpy sont déjà fournis par Colab. Restent les
    # paquets d'imagerie d'app.ai.preprocessing, et pydantic-settings :
    # `app.ai.inference.loader` importe `app.config` pour connaître MODEL_PATH,
    # ce qui tire toute la configuration de l'application. `NoDecode` exige la
    # version 2.7 ou plus récente.
    os.system(
        "pip install --quiet opencv-python-headless pydicom scikit-learn "
        "'pydantic-settings>=2.7'"
    )

if not (REPO_ROOT / "backend" / "app" / "ai").is_dir():
    raise FileNotFoundError(
        f"{REPO_ROOT} ne contient pas backend/app/ai. Le prétraitement doit être "
        "importé du dépôt et non recopié ici : sans lui, le modèle serait entraîné "
        "sur des images préparées autrement qu'à l'inférence."
    )

# `backend/` d'abord : c'est là que vit le paquet `app`.
sys.path.insert(0, str(REPO_ROOT / "backend"))

### Imports et vérification du contrat

Tout ce qui doit rester identique entre l'entraînement et l'inférence est
**importé**, jamais recopié : taille d'entrée, ordre des classes, chaîne de
prétraitement, fabrique du modèle.

In [ ]:
import json
import random
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone

import cv2
import numpy as np
import torch
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset

from app.ai import CLASS_LABELS_FR, CLASS_NAMES, IMAGE_SIZE
from app.ai.inference.loader import build_model, load_checkpoint
from app.ai.inference.predictor import MALIGNANT_INDEX, Predictor
from app.ai.preprocessing.loaders import ImageFormat, detect_format
from app.ai.preprocessing.pipeline import PREPROCESSING_VERSION, preprocess_for_inference
from app.ai.preprocessing.transforms import normalize

# Le contrat de sortie du modèle est figé côté code : le notebook s'y conforme
# au lieu de le redéfinir.
assert CLASS_NAMES == ("benign", "malignant"), CLASS_NAMES
assert CLASS_NAMES[MALIGNANT_INDEX] == "malignant"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("classes            :", CLASS_NAMES)
print("taille d'entrée    :", IMAGE_SIZE)
print("prétraitement      :", PREPROCESSING_VERSION)
print("périphérique       :", DEVICE)

## Étape 1 — Lire `Info.txt`

Format MIAS : `REFNUM BG CLASS SEVERITY X Y RADIUS`.

Trois particularités du fichier réel, vérifiées sur le dataset :

- une image **normale** n'a pas de champ sévérité — la ligne s'arrête après
  `NORM` ;
- une image peut avoir **plusieurs lignes** (plusieurs lésions) : `mdb005` en a
  deux ;
- quatre lignes (`mdb059`, `mdb216`, `mdb233`, `mdb245`) portent une sévérité
  mais **pas de coordonnées** — calcifications diffuses, non localisables. Elles
  restent parfaitement utilisables pour une classification globale de l'image :
  seule la sévérité nous intéresse ici.

In [ ]:
LABEL_NORMAL = "normal"

#: Types d'anomalie du standard MIAS. Une valeur hors de cette liste signale un
#: fichier `Info.txt` qui n'est pas celui attendu.
ABNORMALITY_CLASSES = frozenset({"CALC", "CIRC", "SPIC", "MISC", "ARCH", "ASYM"})

SEVERITY_TO_LABEL = {"B": "benign", "M": "malignant"}


@dataclass(frozen=True)
class MiasLesion:
    """Une ligne d'anomalie de `Info.txt`."""

    refnum: str
    tissue: str
    abnormality: str
    severity: str
    center: tuple[int, int] | None
    radius: int | None


def find_info_file(root: Path) -> Path:
    for candidate in (root / "Info.txt", root / "all-mias" / "Info.txt"):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Info.txt introuvable sous {root}.")


def parse_info(path: Path) -> tuple[dict[str, str], list[MiasLesion]]:
    """Retourne (tissu par image, lésions).

    Toute image citée dans `Info.txt` est connue, y compris les normales : ce
    sont celles qui n'apparaissent dans aucune lésion.
    """
    tissue_by_refnum: dict[str, str] = {}
    lesions: list[MiasLesion] = []

    for number, raw in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        fields = raw.split()
        if not fields or fields[0].upper() == "REFNUM":
            continue
        if len(fields) < 3:
            raise ValueError(f"{path}:{number} : ligne illisible ({raw!r}).")

        refnum, tissue, abnormality = fields[0], fields[1], fields[2].upper()
        tissue_by_refnum.setdefault(refnum, tissue)

        if abnormality == "NORM":
            continue
        if abnormality not in ABNORMALITY_CLASSES:
            raise ValueError(
                f"{path}:{number} : type d'anomalie inconnu {abnormality!r}. "
                "Ce fichier n'est pas un Info.txt mini-MIAS standard."
            )
        if len(fields) < 4:
            raise ValueError(f"{path}:{number} : anomalie sans sévérité ({raw!r}).")

        severity = fields[3].upper()
        if severity not in SEVERITY_TO_LABEL:
            raise ValueError(f"{path}:{number} : sévérité inconnue {severity!r}.")

        # Coordonnées absentes sur les calcifications diffuses : on garde la ligne.
        center, radius = None, None
        if len(fields) >= 7:
            center = (int(fields[4]), int(fields[5]))
            radius = int(fields[6])

        lesions.append(MiasLesion(refnum, tissue, abnormality, severity, center, radius))

    return tissue_by_refnum, lesions


def label_by_refnum(
    tissue_by_refnum: dict[str, str], lesions: list[MiasLesion]
) -> dict[str, str]:
    """Étiquette chaque image : normal, bénin ou malin.

    Une image sans ligne d'anomalie est normale. Une image qui porte à la fois
    une lésion bénigne et une lésion maligne (`mdb144`) est comptée **maligne** :
    l'inverse reviendrait à apprendre au modèle qu'un cancer présent à l'image
    est une image bénigne.
    """
    labels = dict.fromkeys(tissue_by_refnum, LABEL_NORMAL)
    for lesion in lesions:
        label = SEVERITY_TO_LABEL[lesion.severity]
        if labels[lesion.refnum] != "malignant":
            labels[lesion.refnum] = label
    return labels


info_path = find_info_file(MIAS_ROOT)
tissue_by_refnum, lesions = parse_info(info_path)
labels = label_by_refnum(tissue_by_refnum, lesions)

print(f"{info_path} : {len(tissue_by_refnum)} images, {len(lesions)} lésions annotées\n")
for name in (LABEL_NORMAL, "benign", "malignant"):
    count = sum(1 for value in labels.values() if value == name)
    print(f"  {name:10} {count:4}")

conflicts = sorted(
    {lesion.refnum for lesion in lesions if lesion.severity == "M"}
    & {lesion.refnum for lesion in lesions if lesion.severity == "B"}
)
print("\nimages portant les deux sévérités (comptées malignes) :", conflicts or "aucune")

## Étape 2 — Pourquoi les images normales sont écartées

Le modèle déployé a **deux** sorties, dans un ordre figé par le code :
`app.ai.CLASS_NAMES == ("benign", "malignant")`. `predictor.py` lit la
probabilité de l'index 1 comme « probabilité de malignité », `loader.py` refuse
au chargement tout checkpoint dont l'ordre des classes diffère, et l'API expose
`prediction ∈ {benign, malignant}`.

Ajouter une classe `normal` changerait ce contrat de bout en bout : sortie du
réseau, seuil, schémas d'API, base de données, rapports PDF. Ce n'est pas une
décision qui se prend dans un notebook d'entraînement intermédiaire.

Les 207 images normales sont donc **parsées et comptées** — elles font partie de
la lecture d'`Info.txt` — mais **exclues de l'entraînement**. Il reste 115
images. C'est peu, et c'est la principale limite de ce modèle.

In [ ]:
trainable = sorted(refnum for refnum, label in labels.items() if label != LABEL_NORMAL)

if SMOKE_TEST:
    # Échantillon équilibré : la mécanique doit être testée sur les deux classes.
    by_label: dict[str, list[str]] = {"benign": [], "malignant": []}
    for refnum in trainable:
        by_label[labels[refnum]].append(refnum)
    half = SMOKE_MAX_IMAGES // 2
    trainable = sorted(by_label["benign"][:half] + by_label["malignant"][:half])
    print(f"MODE VÉRIFICATION : {len(trainable)} images retenues sur 115.\n")

counts = {name: sum(1 for r in trainable if labels[r] == name) for name in CLASS_NAMES}
print(f"images d'entraînement : {len(trainable)}")
for name in CLASS_NAMES:
    print(f"  {CLASS_LABELS_FR[name]:8} ({name:9}) {counts[name]:4}")

## Étape 3 — Découpage par patiente

Dans mini-MIAS les clichés vont par paires : `mdb001`/`mdb002` sont les seins
gauche et droit de la même patiente, `mdb003`/`mdb004` de la suivante, etc.
Répartir les images au hasard mettrait les deux seins d'une même patiente de
part et d'autre du découpage — le modèle reverrait en test un tissu déjà vu en
entraînement, et les métriques seraient flatteuses et fausses.

L'appariement est vérifié ci-dessous plutôt que supposé : les deux clichés d'une
paire doivent partager la même classe de tissu (`BG`).

In [ ]:
def patient_id(refnum: str) -> int:
    """`mdb001` et `mdb002` → patiente 0, `mdb003` et `mdb004` → patiente 1, …"""
    return (int(refnum.removeprefix("mdb")) - 1) // 2


def check_pairing(tissue_by_refnum: dict[str, str]) -> None:
    """Vérifie l'hypothèse d'appariement sur le tissu mammaire."""
    grouped: dict[int, list[str]] = {}
    for refnum in tissue_by_refnum:
        grouped.setdefault(patient_id(refnum), []).append(refnum)

    mismatched = [
        (patient, refnums)
        for patient, refnums in grouped.items()
        if len({tissue_by_refnum[r] for r in refnums}) > 1
    ]
    print(f"{len(grouped)} patientes pour {len(tissue_by_refnum)} images")
    if mismatched:
        raise ValueError(
            f"{len(mismatched)} paires ont des tissus différents : l'hypothèse "
            f"d'appariement ne tient pas. Exemples : {mismatched[:3]}"
        )
    print("appariement vérifié : chaque paire partage la même classe de tissu")


check_pairing(tissue_by_refnum)

refnums = np.array(trainable)
targets = np.array([CLASS_NAMES.index(labels[r]) for r in trainable])
groups = np.array([patient_id(r) for r in trainable])


def group_split(indices: np.ndarray, n_splits: int) -> tuple[np.ndarray, np.ndarray]:
    """Sépare un bloc en (majorité, 1/n_splits), sans jamais couper une patiente."""
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    major, minor = next(splitter.split(indices, targets[indices], groups[indices]))
    return indices[major], indices[minor]


all_indices = np.arange(len(trainable))
# 5 plis → ~20 % en test ; le reste est redécoupé pour obtenir ~20 % en validation.
train_val_idx, test_idx = group_split(all_indices, n_splits=5 if not SMOKE_TEST else 3)
train_idx, val_idx = group_split(train_val_idx, n_splits=4 if not SMOKE_TEST else 2)

splits = {"train": train_idx, "val": val_idx, "test": test_idx}

for name, indices in splits.items():
    positives = int(targets[indices].sum())
    print(
        f"{name:6} {len(indices):4} images | {positives:3} malignes | "
        f"{len(set(groups[indices])):3} patientes"
    )

# Aucune patiente ne doit apparaître dans deux découpages.
for left in splits:
    for right in splits:
        if left < right:
            shared = set(groups[splits[left]]) & set(groups[splits[right]])
            assert not shared, f"fuite {left}/{right} : patientes {sorted(shared)}"
print("\naucune patiente partagée entre les découpages")

## Étape 4 — Convertir les PGM

Le pipeline d'inférence identifie le format sur la **signature du fichier**, pas
sur son extension : `app.ai.preprocessing.loaders.detect_format` reconnaît PNG,
JPEG et DICOM. Le PGM de MIAS n'en fait pas partie et serait rejeté.

Les images sont donc converties en PNG une fois pour toutes, puis relues comme
n'importe quelle mammographie déposée sur l'API. PNG et non JPEG : la
compression avec pertes introduirait des artefacts sur l'image d'origine.

Les octets sont décodés avec `cv2.imdecode` plutôt que `cv2.imread`, qui échoue
sur les chemins non-ASCII sous Windows.

In [ ]:
def pgm_to_png(source: Path, destination: Path) -> None:
    """Convertit un PGM 8 bits en PNG, sans rien changer aux pixels."""
    buffer = np.frombuffer(source.read_bytes(), dtype=np.uint8)
    image = cv2.imdecode(buffer, cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f"{source} : PGM illisible.")

    success, encoded = cv2.imencode(".png", image)
    if not success:
        raise ValueError(f"{source} : échec de l'encodage PNG.")

    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(encoded.tobytes())


PNG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
png_paths: dict[str, Path] = {}

for refnum in trainable:
    source = MIAS_ROOT / "all-mias" / f"{refnum}.pgm"
    if not source.is_file():
        raise FileNotFoundError(f"{source} : image annoncée par Info.txt mais absente.")

    destination = PNG_CACHE_DIR / f"{refnum}.png"
    if not destination.is_file():
        pgm_to_png(source, destination)
    png_paths[refnum] = destination

# Le pipeline doit reconnaître ces fichiers comme des PNG, sinon l'entraînement
# lirait des images que l'API refuserait.
sample = png_paths[trainable[0]].read_bytes()
assert detect_format(sample) is ImageFormat.PNG, detect_format(sample)

print(f"{len(png_paths)} images converties dans {PNG_CACHE_DIR}")
print(f"format détecté par le pipeline : {detect_format(sample)}")

## Étape 5 — Jeu de données

Chaque image passe par `preprocess_for_inference`, c'est-à-dire **exactement** la
fonction qu'appelle l'API : niveaux de gris → filtre médian → CLAHE →
redimensionnement 384×384 avec remplissage. Rien n'est réimplémenté ici.

Le résultat 8 bits est mis en cache mémoire (115 × 384 × 384 ≈ 17 Mo) : le
prétraitement d'une image 1024×1024 coûte assez cher pour ne pas le refaire à
chaque époque.

L'augmentation ne s'applique qu'au jeu d'entraînement, et **après** le
prétraitement, sur l'image 8 bits — puis `normalize` termine la chaîne comme à
l'inférence. Deux transformations seulement, géométriques :

- miroir horizontal, qui revient à changer de côté ;
- rotation et zoom légers, qui simulent les variations de positionnement.

Pas de jitter d'intensité : CLAHE vient précisément d'égaliser le contraste, le
perturber ensuite reviendrait à défaire l'étape de prétraitement.

In [ ]:
MAX_ROTATION_DEGREES = 10.0
MAX_ZOOM = 0.10


def augment(image: np.ndarray, rng: random.Random) -> np.ndarray:
    """Miroir et petite transformation affine sur l'image 8 bits prétraitée."""
    if rng.random() < 0.5:
        image = cv2.flip(image, 1)

    angle = rng.uniform(-MAX_ROTATION_DEGREES, MAX_ROTATION_DEGREES)
    scale = 1.0 + rng.uniform(-MAX_ZOOM, MAX_ZOOM)
    height, width = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, scale)

    # Remplissage noir : c'est déjà la valeur du fond sur une mammographie.
    return cv2.warpAffine(
        image, matrix, (width, height), flags=cv2.INTER_LINEAR, borderValue=0
    )


class MiasDataset(Dataset):
    """Images mini-MIAS prétraitées par la chaîne d'inférence."""

    def __init__(self, indices: np.ndarray, *, training: bool) -> None:
        self.refnums = [refnums[i] for i in indices]
        self.targets = [int(targets[i]) for i in indices]
        self.training = training
        self.rng = random.Random(SEED)

        self.images: list[np.ndarray] = []
        for refnum in self.refnums:
            preprocessed = preprocess_for_inference(png_paths[refnum].read_bytes())
            # `display` est l'image 8 bits que le modèle voit, avant normalisation.
            self.images.append(preprocessed.display)

    def __len__(self) -> int:
        return len(self.refnums)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        image = self.images[index]
        if self.training:
            image = augment(image, self.rng)
        tensor = torch.from_numpy(np.ascontiguousarray(normalize(image)))
        return tensor, self.targets[index]


datasets = {
    name: MiasDataset(indices, training=(name == "train"))
    for name, indices in splits.items()
}

loaders = {
    name: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=(name == "train"),
        num_workers=0,  # Colab : les workers coûtent plus qu'ils ne rapportent ici.
        drop_last=False,
    )
    for name, dataset in datasets.items()
}

batch, batch_targets = next(iter(loaders["train"]))
print("lot d'entraînement :", tuple(batch.shape), batch.dtype)
print("cibles             :", batch_targets.tolist())
assert batch.shape[1:] == (3, *IMAGE_SIZE), batch.shape

## Étape 6 — Modèle et entraînement

Le réseau est construit par `app.ai.inference.loader.build_model` : la même
fabrique que celle du chargement, donc une architecture nécessairement
compatible avec le checkpoint produit.

Deux taux d'apprentissage : la tête part de zéro et apprend vite, le corps
pré-entraîné se contente d'un ajustement fin. Avec aussi peu d'images, une seule
valeur élevée détruirait les représentations ImageNet.

La perte est pondérée par l'inverse de la fréquence des classes, et le meilleur
modèle est retenu sur le **rappel malin** en validation, pas sur l'accuracy :
manquer un cancer est la défaillance la plus grave du système.

Le ROC-AUC sert de départage. Le rappel seul se maximise trivialement en
répondant « malin » à tout : un tel modèle atteindrait 1,00 dès la première
époque et resterait sélectionné jusqu'au bout sans rien discriminer.

In [ ]:
model = build_model(ARCHITECTURE, pretrained=True).to(DEVICE)

# La tête (créée par build_model) apprend vite ; le corps ImageNet est ajusté
# doucement.
head_parameters = list(model.classifier.parameters())
head_ids = {id(p) for p in head_parameters}
backbone_parameters = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_parameters, "lr": LR_BACKBONE},
        {"params": head_parameters, "lr": LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Pondération inverse de la fréquence : le déséquilibre bénin/malin est modéré
# ici, mais le laisser jouer tirerait le modèle vers la classe majoritaire.
train_targets = targets[splits["train"]]
frequencies = np.bincount(train_targets, minlength=len(CLASS_NAMES)).astype(np.float64)
weights = frequencies.sum() / (len(CLASS_NAMES) * np.maximum(frequencies, 1.0))
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE)
)

print("effectifs d'entraînement :", dict(zip(CLASS_NAMES, frequencies.astype(int))))
print("pondération de la perte  :", dict(zip(CLASS_NAMES, weights.round(3))))

In [ ]:
@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    """Retourne (cibles, probabilités de malignité) pour un jeu complet."""
    model.eval()
    all_targets: list[int] = []
    all_scores: list[float] = []

    for inputs, batch_targets in loader:
        probabilities = torch.softmax(model(inputs.to(DEVICE)), dim=1)
        all_scores.extend(probabilities[:, MALIGNANT_INDEX].cpu().tolist())
        all_targets.extend(batch_targets.tolist())

    return np.array(all_targets), np.array(all_scores)


def malignant_recall(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> float:
    predictions = (scores >= threshold).astype(int)
    return float(recall_score(y_true, predictions, pos_label=1, zero_division=0))


best_score = (-1.0, -1.0)
best_state: dict[str, torch.Tensor] | None = None
best_epoch = 0
history: list[dict[str, float]] = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for inputs, batch_targets in loaders["train"]:
        inputs = inputs.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(inputs), batch_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    scheduler.step()

    train_loss = running_loss / len(datasets["train"])
    val_targets, val_scores = evaluate(loaders["val"])
    recall = malignant_recall(val_targets, val_scores, 0.5)
    # ROC-AUC indéfini si la validation ne contient qu'une seule classe.
    auc = (
        float(roc_auc_score(val_targets, val_scores))
        if len(set(val_targets.tolist())) > 1
        else float("nan")
    )

    history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_recall": recall, "val_auc": auc}
    )
    print(
        f"époque {epoch:3} | perte {train_loss:.4f} | "
        f"rappel malin (val) {recall:.3f} | ROC-AUC (val) {auc:.3f}"
    )

    # Le rappel malin décide, le ROC-AUC départage. Sans ce second critère, un
    # modèle qui répondrait « malin » à tout obtiendrait un rappel de 1,0 dès la
    # première époque et resterait sélectionné jusqu'à la fin, alors qu'il ne
    # discrimine rien.
    score = (recall, 0.0 if np.isnan(auc) else auc)
    if score > best_score:
        best_score = score
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:
        print(f"arrêt anticipé : aucun progrès depuis l'époque {best_epoch}")
        break

assert best_state is not None, "aucune époque n'a été exécutée"
model.load_state_dict(best_state)
print(
    f"\nmeilleur état retenu : époque {best_epoch} "
    f"(rappel malin {best_score[0]:.3f}, ROC-AUC {best_score[1]:.3f})"
)

#: ROC-AUC de validation de l'époque retenue. Sert de témoin : le modèle évalué
#: puis sauvegardé plus bas doit reproduire cette valeur, sinon ce ne sont plus
#: les mêmes poids.
BEST_VAL_AUC = float(history[best_epoch - 1]["val_auc"])


def restore_best_state() -> None:
    """Réapplique au modèle en mémoire les poids de l'époque retenue.

    Appelée en tête de chaque cellule qui évalue ou sauvegarde, ce qui rend la
    fin du notebook indifférente à l'ordre d'exécution des cellules.

    Sans elle, ré-exécuter la cellule précédente — celle qui fait
    `build_model(ARCHITECTURE, pretrained=True)` — remplace le modèle entraîné
    par un réseau ImageNet neuf, et tout ce qui suit porte sur ce réseau : les
    métriques de test, le checkpoint écrit, le modèle déployé. C'est arrivé, et
    rien ne l'a signalé : les métriques de validation, calculées avant la
    ré-exécution, restaient bonnes dans la fiche du modèle.
    """
    assert best_state is not None, "aucune époque sélectionnée : relancer l'entraînement"
    model.load_state_dict(best_state)

## Étape 7 — Choix du seuil

Le seuil n'est pas laissé à 0,5 : il est choisi sur la validation comme le seuil
**le plus haut** qui atteint encore la sensibilité visée. À sensibilité donnée,
c'est celui qui produit le moins de faux positifs.

Si aucun seuil n'atteint la cible — cas plausible avec une validation de cette
taille — on retombe sur l'indice de Youden, et le notebook le signale.

In [ ]:
def choose_threshold(
    y_true: np.ndarray, scores: np.ndarray, target_sensitivity: float
) -> tuple[float, str]:
    """Seuil le plus élevé atteignant la sensibilité cible, sinon Youden."""
    candidates = np.unique(np.concatenate([scores, [0.0, 1.0]]))

    achieving = [
        threshold
        for threshold in candidates
        if malignant_recall(y_true, scores, float(threshold)) >= target_sensitivity
    ]
    if achieving:
        threshold = float(max(achieving))
        return threshold, (
            f"seuil le plus élevé atteignant une sensibilité ≥ {target_sensitivity:.2f} "
            "sur la validation"
        )

    def youden(threshold: float) -> float:
        predictions = (scores >= threshold).astype(int)
        matrix = confusion_matrix(y_true, predictions, labels=[0, 1])
        true_negative, false_positive, false_negative, true_positive = matrix.ravel()
        sensitivity = true_positive / max(true_positive + false_negative, 1)
        specificity = true_negative / max(true_negative + false_positive, 1)
        return sensitivity + specificity - 1

    threshold = float(max(candidates, key=lambda t: youden(float(t))))
    return threshold, (
        f"sensibilité de {target_sensitivity:.2f} inatteignable sur la validation : "
        "seuil retenu par l'indice de Youden"
    )


# Les poids évalués sont ceux de l'époque retenue, quoi qu'il ait été
# ré-exécuté entre-temps.
restore_best_state()

val_targets, val_scores = evaluate(loaders["val"])
threshold, threshold_rationale = choose_threshold(
    val_targets, val_scores, TARGET_SENSITIVITY
)

print(f"seuil retenu : {threshold:.4f}")
print(f"raison       : {threshold_rationale}")
print(f"sensibilité en validation : {malignant_recall(val_targets, val_scores, threshold):.3f}")

## Étape 8 — Évaluation sur le jeu de test

Le jeu de test compte une vingtaine d'images. **Chaque erreur pèse cinq points
d'accuracy** : ces chiffres situent un ordre de grandeur, ils ne mesurent pas une
performance. C'est une raison de plus de remplacer ce modèle par un entraînement
sur Mini-DDSM.

In [ ]:
def compute_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    predictions = (scores >= threshold).astype(int)
    matrix = confusion_matrix(y_true, predictions, labels=[0, 1])
    true_negative, false_positive, false_negative, true_positive = (
        int(value) for value in matrix.ravel()
    )

    return {
        "n_images": int(len(y_true)),
        "accuracy": float((predictions == y_true).mean()),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall_malignant": float(recall_score(y_true, predictions, zero_division=0)),
        "specificity": float(true_negative / max(true_negative + false_positive, 1)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": (
            float(roc_auc_score(y_true, scores))
            if len(set(y_true.tolist())) > 1
            else None
        ),
        "confusion_matrix": {
            "true_negative": true_negative,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "true_positive": true_positive,
        },
    }


restore_best_state()

test_targets, test_scores = evaluate(loaders["test"])
test_metrics = compute_metrics(test_targets, test_scores, threshold)
val_metrics = compute_metrics(val_targets, val_scores, threshold)

for key, value in test_metrics.items():
    if key != "confusion_matrix":
        print(f"{key:18} {value}")

matrix = test_metrics["confusion_matrix"]
print("\n                 prédit bénin  prédit malin")
print(f"réel bénin       {matrix['true_negative']:12}  {matrix['false_positive']:12}")
print(f"réel malin       {matrix['false_negative']:12}  {matrix['true_positive']:12}")

if matrix["false_negative"]:
    print(
        f"\n⚠️ {matrix['false_negative']} cancer(s) manqué(s) sur "
        f"{matrix['false_negative'] + matrix['true_positive']} — "
        "c'est la défaillance la plus grave du système."
    )

# Un modèle qui répond la même chose à tout obtient d'excellentes métriques sur
# une des deux classes. Mieux vaut que le notebook le dise que de laisser lire
# un rappel de 1,00 comme une réussite.
#
# Tolérance, et non égalité stricte à zéro : lors de l'incident du 15/08 la
# spécificité valait 0,0019 — un cliché bénin sur 536 tombé du bon côté — et
# `== 0.0` n'a rien déclenché sur un modèle qui répondait pourtant « malin » à
# tout. Un flottant issu d'un comptage ne vaut presque jamais exactement zéro.
DEGENERATE_TOLERANCE = 0.01

is_degenerate = (
    test_metrics["specificity"] <= DEGENERATE_TOLERANCE
    or test_metrics["recall_malignant"] <= DEGENERATE_TOLERANCE
)
if is_degenerate:
    print(
        "\n⛔ MODÈLE DÉGÉNÉRÉ : toutes les images ou presque reçoivent la même "
        "classe au seuil retenu. Ce checkpoint ne discrimine rien et ne doit pas "
        "être déployé, quelles que soient les autres métriques. L'étape 9 "
        "refusera de l'écrire."
    )

## Étape 9 — Enregistrement du checkpoint

Le format est celui qu'attend `app.ai.inference.loader` :

- `class_names` repris de `app.ai.CLASS_NAMES`, dans l'ordre figé ;
- `preprocessing_version` repris de `app.ai.preprocessing` ;
- `version` **sans** le préfixe `placeholder-`, réservé aux modèles de
  substitution et refusé par le chargeur pour un modèle entraîné ;
- `clinically_validated` à **`False`**, explicitement. Ce modèle cesse d'être un
  placeholder, mais cela ne dit rien de sa valeur clinique : l'API continuera
  d'afficher un avertissement, différent de celui du placeholder. Rien dans ce
  notebook — aucune métrique, aussi bonne soit-elle — n'autorise à passer ce
  drapeau à `True`.

Les clés supplémentaires sont ignorées par le chargeur mais lues par les
humains : elles évitent qu'un `.pt` retrouvé dans six mois soit un fichier de
poids anonyme. Une fiche modèle JSON les reprend, comme le demande
`models/README.md`.

### Trois contrôles avant d'écrire

Le 15 août 2026, ce notebook a produit un checkpoint contenant un réseau
ImageNet intact, sous des métadonnées annonçant treize époques et une époque
sélectionnée. La cellule de l'étape 6 avait été ré-exécutée après
l'entraînement : `model` était redevenu un `build_model(pretrained=True)` neuf,
et `torch.save` a écrit cet état. Le fichier a passé toutes les validations de
contrat et a été déployé, où il répondait 50 % à toute image.

Trois contrôles ferment cette porte, et **aucun checkpoint n'est écrit s'ils
échouent** :

| Contrôle | Ce qu'il attrape |
|----------|------------------|
| Corps du réseau ≠ ImageNet | un modèle jamais entraîné, quoi qu'en disent ses métadonnées |
| ROC-AUC recalculé ≈ celui de l'époque retenue | des poids qui ne sont plus ceux qui ont été sélectionnés |
| Modèle non dégénéré | un réseau qui répond la même classe à tout |

S'y ajoute un changement de fond : ce qui est sauvegardé est `best_state`, l'état
sélectionné à l'étape 6, et non `model.state_dict()` — l'état que le modèle en
mémoire se trouve avoir. Les cellules d'évaluation appellent par ailleurs
`restore_best_state()`, ce qui rend toute la fin du notebook indifférente à
l'ordre d'exécution des cellules.

In [ ]:
def git_commit(repo: Path) -> str | None:
    try:
        result = subprocess.run(
            ["git", "-C", str(repo), "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
    except (OSError, subprocess.CalledProcessError):
        return None
    return result.stdout.strip()


assert not MODEL_VERSION.startswith("placeholder-"), (
    "le préfixe placeholder- est réservé aux modèles de substitution"
)

# --------------------------------------------------------------------------- #
# Contrôles d'intégrité — rien n'est écrit tant qu'ils ne sont pas passés.
#
# Les métadonnées de cette cellule décrivent ce que le notebook *croit*
# sauvegarder. Elles ne prouvent rien sur les poids réellement écrits : un
# checkpoint annonçant treize époques et une époque retenue a déjà été produit
# à partir d'un réseau ImageNet intact, et il a été déployé.
# --------------------------------------------------------------------------- #

#: Écart maximal toléré entre le ROC-AUC recalculé avant sauvegarde et celui de
#: l'époque sélectionnée. L'évaluation étant déterministe et la validation sans
#: augmentation, les deux valeurs devraient être identiques ; la marge ne couvre
#: que le non-déterminisme des noyaux CUDA.
SELECTION_AUC_TOLERANCE = 0.01

restore_best_state()


def backbone_matches_imagenet(state_dict: dict[str, torch.Tensor]) -> bool:
    """Le corps du réseau est-il resté exactement celui d'ImageNet ?

    Un seul pas de descente de gradient modifie tous les poids du corps. Un
    corps identique bit à bit signifie donc qu'aucun entraînement n'a eu lieu,
    quelles que soient les métriques calculées plus haut. La tête est exclue :
    `build_model` la remplace à chaque construction, elle diffère de la
    référence même sur un réseau vierge.
    """
    reference = build_model(ARCHITECTURE, pretrained=True).state_dict()
    for name, tensor in reference.items():
        if name.startswith("classifier."):
            continue
        saved = state_dict.get(name)
        if saved is None or not torch.equal(saved.cpu(), tensor.cpu()):
            return False
    return True


if backbone_matches_imagenet(best_state):
    raise ValueError(
        "le corps du réseau est identique aux poids ImageNet : l'entraînement "
        "n'y a laissé aucune trace, ce checkpoint répondrait 50 % à toute "
        "image. Cause habituelle : la cellule de l'étape 6 qui construit le "
        "modèle a été ré-exécutée après l'entraînement. Relancer le notebook de "
        "haut en bas, sans ré-exécution partielle."
    )

# Le modèle sur le point d'être écrit doit reproduire la mesure qui l'a fait
# sélectionner. Sinon, ce ne sont plus les poids de l'époque retenue.
selection_targets, selection_scores = evaluate(loaders["val"])
if not np.isnan(BEST_VAL_AUC) and len(set(selection_targets.tolist())) > 1:
    selection_auc = float(roc_auc_score(selection_targets, selection_scores))
    if abs(selection_auc - BEST_VAL_AUC) > SELECTION_AUC_TOLERANCE:
        raise ValueError(
            f"ROC-AUC de validation {selection_auc:.4f} au moment d'écrire, contre "
            f"{BEST_VAL_AUC:.4f} à l'époque {best_epoch} qui a été sélectionnée. "
            "Les poids en mémoire ne sont pas ceux qui ont été choisis : ne rien "
            "écrire tant que l'écart n'est pas expliqué."
        )
    print(
        f"contrôle de sélection : ROC-AUC {selection_auc:.4f} "
        f"≈ {BEST_VAL_AUC:.4f} (époque {best_epoch})"
    )

if is_degenerate:
    if SMOKE_TEST:
        # Une époque sur une poignée d'images produit presque toujours un modèle
        # dégénéré : bloquer ici empêcherait de vérifier la mécanique du
        # notebook, ce qui est précisément l'objet du mode vérification.
        print(
            "mode vérification : modèle dégénéré toléré, le checkpoint produit "
            "n'a aucune vocation à être déployé."
        )
    else:
        raise ValueError(
            "modèle dégénéré (voir l'étape 8) : toutes les images ou presque "
            "reçoivent la même classe. Un tel checkpoint ne doit pas atteindre "
            "models/, où il serait indiscernable d'un modèle utile."
        )

normal_count = sum(1 for value in labels.values() if value == LABEL_NORMAL)

LIMITATIONS = [
    f"Entraîné sur {len(trainable)} images lésionnelles de mini-MIAS "
    f"({len(labels)} clichés au total, dont {normal_count} normaux écartés faute "
    "de classe `normal` dans le contrat de sortie).",
    f"Jeu de test de {len(splits['test'])} images : les métriques donnent un "
    "ordre de grandeur, pas une performance mesurée.",
    "mini-MIAS est un corpus de films numérisés des années 1990, non représentatif "
    "des mammographes numériques actuels.",
    "Aucune validation sur une population externe, ni sur les tranches d'âge, "
    "densités mammaires ou constructeurs réels d'utilisation.",
    "Modèle de transition : à remplacer par un entraînement Mini-DDSM.",
]

metadata = {
    "architecture": ARCHITECTURE,
    "class_names": list(CLASS_NAMES),
    "preprocessing_version": PREPROCESSING_VERSION,
    "threshold": float(threshold),
    "version": MODEL_VERSION,
    # Explicite, alors que le chargeur retomberait de toute façon sur False en
    # l'absence de clé : ce modèle est entraîné mais n'a fait l'objet d'aucune
    # revue clinique. Le passer à True demande une validation humaine documentée
    # menée hors de ce notebook — voir DEFAULT_CLINICALLY_VALIDATED dans
    # app/ai/inference/loader.py. Aucun résultat de cette cellule ne l'autorise.
    "clinically_validated": False,
    "image_size": list(IMAGE_SIZE),
    "dataset": {
        "name": "mini-MIAS",
        "source": "http://peipa.essex.ac.uk/info/mias.html",
        "images_total": len(labels),
        "images_normal_excluded": normal_count,
        "images_used": len(trainable),
        "class_counts": counts,
    },
    "split": {
        "strategy": "StratifiedGroupKFold par patiente (paires MIAS)",
        "seed": SEED,
        **{name: int(len(indices)) for name, indices in splits.items()},
    },
    "training": {
        "epochs_run": len(history),
        "best_epoch": best_epoch,
        "batch_size": BATCH_SIZE,
        "lr_head": LR_HEAD,
        "lr_backbone": LR_BACKBONE,
        "weight_decay": WEIGHT_DECAY,
        "class_weights": dict(zip(CLASS_NAMES, weights.tolist())),
        "selection_criterion": (
            "rappel sur la classe maligne en validation, ROC-AUC en départage"
        ),
        "augmentation": "miroir horizontal, rotation ±10°, zoom ±10 %",
        "smoke_test": SMOKE_TEST,
    },
    "threshold_rationale": threshold_rationale,
    "metrics": {"validation": val_metrics, "test": test_metrics},
    "limitations": LIMITATIONS,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "git_commit": git_commit(REPO_ROOT),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stem = f"breastai_{ARCHITECTURE}_mini-mias-v1_{datetime.now().strftime('%Y%m%d')}"
checkpoint_path = OUTPUT_DIR / f"{stem}.pt"
model_card_path = OUTPUT_DIR / f"{stem}.json"

# `best_state` et non `model.state_dict()` : ce qui est écrit est exactement
# l'état sélectionné à l'étape 6, pas ce que le modèle en mémoire se trouve
# contenir au moment où cette cellule s'exécute.
torch.save({**metadata, "state_dict": best_state}, checkpoint_path)
model_card_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), "utf-8")

print(f"checkpoint  : {checkpoint_path}")
print(f"fiche modèle: {model_card_path}")

## Étape 10 — Relire le checkpoint avec le code de production

Le notebook ne se termine pas sur un `torch.save`. Le fichier est rechargé par
**`load_checkpoint`**, celui-là même qu'exécute l'API au démarrage, puis une
prédiction est produite via `Predictor`. Un contrat cassé échoue ici, sur le
poste qui a entraîné le modèle, et non au démarrage du serveur.

S'y ajoute une comparaison de non-régression : sur un échantillon témoin de la
validation, le modèle rechargé depuis le disque doit produire les mêmes scores
que celui resté en mémoire. Un écart de ROC-AUC supérieur à 0,02 lève une
erreur.

Ce contrôle couvre la **sérialisation** — poids manquants, tampons de BatchNorm
perdus, chargement partiel. Il ne remplace pas ceux de l'étape 9, et il faut
être clair sur sa portée : lors de l'incident du 15 août, mémoire et disque
étaient parfaitement d'accord, tous deux sur un réseau ImageNet. Un checkpoint
peut être fidèlement écrit *et* faux. C'est l'étape 9 qui attrape ce cas-là.

In [ ]:
bundle = load_checkpoint(checkpoint_path, torch.device("cpu"))

assert bundle.is_placeholder is False, "le modèle ne doit pas être vu comme placeholder"
assert bundle.class_names == CLASS_NAMES
assert bundle.preprocessing_version == PREPROCESSING_VERSION
assert bundle.architecture == ARCHITECTURE
assert abs(bundle.threshold - threshold) < 1e-9
# Ne plus être un placeholder ne vaut pas validation clinique : c'est
# précisément la confusion que le drapeau sépare.
assert bundle.clinically_validated is False

print(f"version           : {bundle.version}")
print(f"architecture      : {bundle.architecture}")
print(f"classes           : {bundle.class_names}")
print(f"seuil             : {bundle.threshold:.4f}")
print(f"prétraitement     : {bundle.preprocessing_version}")
print(f"placeholder       : {bundle.is_placeholder}")
print(f"validé cliniquement : {bundle.clinically_validated}")

# --------------------------------------------------------------------------- #
# Non-régression : le modèle rechargé doit répondre comme celui en mémoire.
#
# Ce contrôle couvre la sérialisation — poids manquants, tampons de BatchNorm
# perdus, chargement partiel. Il ne remplace pas ceux de l'étape 9 : lors de
# l'incident du 15/08, mémoire et disque étaient d'accord, tous deux sur un
# réseau ImageNet. Un checkpoint peut être fidèlement écrit *et* faux.
# --------------------------------------------------------------------------- #
RELOAD_AUC_TOLERANCE = 0.02
PROBE_SIZE = 64

probe_subset = torch.utils.data.Subset(
    datasets["val"], list(range(min(PROBE_SIZE, len(datasets["val"]))))
)
probe_loader = DataLoader(probe_subset, batch_size=BATCH_SIZE, shuffle=False)


@torch.no_grad()
def scores_of(network: nn.Module, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    """Cibles et probabilités de malignité d'un réseau sur l'échantillon témoin."""
    network.eval()
    seen_targets: list[int] = []
    seen_scores: list[float] = []
    for inputs, batch_targets in probe_loader:
        probabilities = torch.softmax(network(inputs.to(device)), dim=1)
        seen_scores.extend(probabilities[:, MALIGNANT_INDEX].cpu().tolist())
        seen_targets.extend(batch_targets.tolist())
    return np.array(seen_targets), np.array(seen_scores)


restore_best_state()
memory_targets, memory_scores = scores_of(model, DEVICE)
reloaded_targets, reloaded_scores = scores_of(bundle.model, bundle.device)

assert (memory_targets == reloaded_targets).all(), "échantillon témoin désaligné"

largest_gap = float(np.max(np.abs(memory_scores - reloaded_scores)))
print(f"\nécart maximal de score sur {len(memory_scores)} images : {largest_gap:.6f}")

if len(set(memory_targets.tolist())) > 1:
    memory_auc = float(roc_auc_score(memory_targets, memory_scores))
    reloaded_auc = float(roc_auc_score(reloaded_targets, reloaded_scores))
    print(f"ROC-AUC mémoire {memory_auc:.4f} | rechargé {reloaded_auc:.4f}")
    if abs(memory_auc - reloaded_auc) > RELOAD_AUC_TOLERANCE:
        raise ValueError(
            f"ROC-AUC {memory_auc:.4f} en mémoire contre {reloaded_auc:.4f} après "
            "rechargement du checkpoint qui vient d'être écrit. Le fichier ne "
            "contient pas le modèle évalué : ne pas le déployer."
        )
else:
    print("échantillon témoin monoclasse : comparaison sur les scores seuls.")


# Une prédiction complète, du fichier PNG au résultat rendu par l'API.
probe_refnum = refnums[splits["test"][0]]
probe = preprocess_for_inference(png_paths[probe_refnum].read_bytes())
result = Predictor(bundle).predict(probe.tensor)

print(f"\nimage {probe_refnum} (vérité terrain : {labels[probe_refnum]})")
print(f"  prédiction  : {result.label}")
print(f"  p(malin)    : {result.probability:.4f}")
print(f"  confiance   : {result.confidence:.4f}")
print(f"  version     : {result.model_version}")

# Grad-CAM tourne sur la dernière couche convolutive : elle doit exister.
assert bundle.target_layer is not None
print("\ncontrat de checkpoint vérifié de bout en bout")

## Déployer, et après

```bash
cp models/breastai_efficientnet_b0_mini-mias-v1_<date>.pt models/breastai_efficientnet.pt
```

`MODEL_PATH` pointe dessus par défaut. Aucun autre changement n'est nécessaire :
ni l'inférence, ni les services, ni l'API ne bougent. Les analyses déjà rendues
par le placeholder restent identifiables — leur `model_version` garde le préfixe
`placeholder-` — et peuvent être rejouées par
`POST /analyses/{id}/infer`.

Une fois ce modèle déployé, `is_placeholder_model` passe à `false` mais
`clinically_validated` reste `false` : l'interface et les rapports remplacent le
bandeau « modèle démo » par « modèle entraîné mais non validé cliniquement ».
Le bandeau ne disparaît pas — il change.

### Ce que ce modèle n'est pas

Il a vu 115 mammographies numérisées, toutes issues d'un corpus des années 1990.
Il n'a jamais vu de cliché normal, puisque le contrat de sortie ne comporte pas
cette classe : présenté à un sein sain, il répondra `benign` ou `malignant`, sans
troisième possibilité. Ses métriques sont mesurées sur une vingtaine d'images.

**L'avertissement clinique reste inchangé** : aucune décision médicale ne doit
s'appuyer sur ces prédictions. Le passage à Mini-DDSM reste la prochaine étape,
et ce modèle est là pour être remplacé.